<a href="https://colab.research.google.com/github/AnnesaAD05/Python-Projects-Informatics-1st-year-annesa/blob/main/ollama_project_Delhi_Darshan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📦 1. Dependency Installation
We install `streamlit`, `gTTS`, `python-dotenv`, `requests`, and the `ollama` Python bindings.

In [1]:
!pip install streamlit gTTS python-dotenv requests ollama --quiet
!npm install -g localtunnel --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠇  run `npm fund` for details
⠇

## ⚙️ 2. Environment Configuration
We create a `.env` template file to handle configurations.

In [2]:
with open('.env', 'w') as f:
    f.write('''# Delhi Darshan App Configuration
OPENWEATHER_API_KEY=
OLLAMA_MODEL=llama3.2
''')

## ☀️ 3. Weather Integration (`weather.py`)
Defines logic for fetching live OpenWeather metrics with a reliable simulator fallback if no API key is specified.

In [3]:
%%writefile weather.py
import os
import requests
import random

def get_live_weather(lat, lon, api_key=None):
    if not api_key:
        api_key = os.getenv("OPENWEATHER_API_KEY", "")

    if not api_key:
        # High quality simulated realistic Delhi weather values
        temp = round(random.uniform(22.0, 34.0), 1)
        humidity = random.randint(45, 80)
        wind = round(random.uniform(2.0, 8.0), 1)
        desc = random.choice(["Clear Sky", "Haze", "Partly Cloudy", "Warm Breeze"])
        return {
            "temp": temp,
            "humidity": humidity,
            "wind_speed": wind,
            "description": desc,
            "icon": "01d",
            "simulated": True
        }

    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}&units=metric"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return {
                "temp": data["main"]["temp"],
                "humidity": data["main"]["humidity"],
                "wind_speed": data["wind"]["speed"],
                "description": data["weather"][0]["description"].capitalize(),
                "icon": data["weather"][0]["icon"],
                "simulated": False
            }
    except Exception:
        pass

    # Emergency fallback
    return {
        "temp": 28.5,
        "humidity": 60,
        "wind_speed": 4.2,
        "description": "Hazy Sun (Fallback)",
        "icon": "50d",
        "simulated": True
    }

def get_comfort_recommendation(temp, description):
    if temp > 35:
        return "🔴 Extremely Hot: Wear light clothes, carry a water flask, and prefer indoor visits."
    elif temp > 28:
        return "🟡 Warm conditions: Ideal for early morning/late evening garden or monument walks."
    elif temp < 15:
        return "🔵 Quite Cool: Carry a light jacket or shawl; perfect for daytime explorations!"
    else:
        return "🟢 Excellent Comfort: Perfect climate for spending quality outdoor time."

Writing weather.py


## 🚦 4. Crowd Engine Module (`crowd.py`)
Computes live crowd estimations based on the current hour of the day, peak hours, weekends, and holiday multipliers.

In [4]:
%%writefile crowd.py
from datetime import datetime

def estimate_crowd_level(current_hour=None, is_weekend=False, place_name=None, *args, **kwargs):
    if current_hour is None:
        now = datetime.now()
        current_hour = now.hour
        is_weekend = now.weekday() >= 5

    # Realistic time-based crowd simulation (fully offline)
    if current_hour < 8 or current_hour > 20:
        return "🟢 Low Crowd", "Monument Closed / Quiet hours. Safe for exterior peaceful views."

    if 11 <= current_hour <= 15:
        level = "🔴 Peak Crowd" if is_weekend else "🟡 Moderate Crowd"
        rec = "Busy hours with tourist lines. Stay hydrated and book online tickets."
    elif 16 <= current_hour <= 19:
        level = "🔴 Peak Crowd"
        rec = "Sunset hour rush! High peak visitor movement. Great for photography but expects queues."
    else:
        level = "🟢 Low Crowd"
        rec = "Excellent quiet window to explore monuments in relative peace."

    return level, rec

Writing crowd.py


## 🤖 5. Ollama & AI Narration Helper (`ollama_guide.py`)
Responsible for communicating with Ollama client with fallback options to ensure 100% uptime.

In [5]:
%%writefile ollama_guide.py
import os
import ollama

def generate_monument_insights(name, era, default_info):
    model = os.getenv("OLLAMA_MODEL", "llama3.2")
    prompt = f"""You are an expert Delhi Darshan AI tour guide.
    Provide an immersive, highly engaging, brief narrative about: '{name}' from the {era} era.
    Format your response inside three clearly separated markdown sections:

    ### 📜 History & Legends
    (A rich cultural backstory)

    ### 🏛️ Architectural Wonders
    (Unique details about builders, domes, or material)

    ### 🍢 Street Food & Surrounds
    (Must-visit local markets, context, or famous snacks like Chat/Paranthe nearby)
    """
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        return response['message']['content']
    except Exception:
        # Elegant structured custom offline fallback generator
        return f"""### 📜 History & Legends
{default_info}

### 🏛️ Architectural Wonders
Built with characteristic Delhi-centric materials and structural designs reflecting the transition across classical Indian, Islamic, and Colonial eras.

### 🍢 Street Food & Surrounds
Step outside to enjoy delicious North Indian delicacies, street food chats, sweet Jalebis, or warm chai in nearby heritage lanes!"""

Writing ollama_guide.py


## 🔊 6. Text-To-Speech Audio Module (`audio.py`)
Converts guided scripts to audio formats directly downloadable/playable inside Streamlit.

In [6]:
%%writefile audio.py
from gtts import gTTS
import io

def text_to_mp3_bytes(text):
    try:
        # Clean text for high quality short summary voice over
        clean_text = text.replace("#", "").replace("*", "").split("\n")[0:6]
        voice_text = " ".join(clean_text)
        if len(voice_text.strip()) < 10:
            voice_text = "Welcome to the Delhi Darshan Audio tour companion."

        tts = gTTS(text=voice_text, lang='en', tld='co.in', slow=False)
        fp = io.BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)
        return fp.getvalue()
    except Exception:
        return None

Writing audio.py


## 🌟 7. Main Web Application (`app.py`)
Our primary user dashboard showing widgets, filters, maps, and chats.

In [7]:
%%writefile app.py
import streamlit as st
import os
from dotenv import load_dotenv
from datetime import datetime

# Load environmental variables
load_dotenv()

# Import modular helpers
from weather import get_live_weather, get_comfort_recommendation
from crowd import estimate_crowd_level
from ollama_guide import generate_monument_insights
from audio import text_to_mp3_bytes

st.set_page_config(page_title="Delhi Darshan AI Guide", page_icon="🕌", layout="wide")

# 20 Curated Monuments
monuments_db = [
    {
        "name": "Red Fort (Lal Qila)",
        "builder": "Shah Jahan",
        "year": "1648",
        "era": "Mughal",
        "lat": 28.6562, "lon": 77.2410,
        "metro": "Lal Qila (Violet Line) / Chandni Chowk (Yellow Line)",
        "last_mile": "🚲 E-Rickshaw or 10-minute historical bazaar walk",
        "overview": "Constructed as the fortified palace of Shahjahanabad, serving as the throne of Mughal rule for 200 years."
    },
    {
        "name": "Qutub Minar",
        "builder": "Qutb-ud-din Aibak",
        "year": "1199",
        "era": "Delhi Sultanate",
        "lat": 28.5245, "lon": 77.1855,
        "metro": "Qutub Minar (Yellow Line)",
        "last_mile": "🛺 Auto-Rickshaw directly to the ticket complex (5 mins)",
        "overview": "A magnificent 73-meter tall victory minaret celebrating early Indo-Islamic architectural style."
    },
    {
        "name": "Humayun's Tomb",
        "builder": "Empress Bega Begum",
        "year": "1570",
        "era": "Mughal",
        "lat": 28.5933, "lon": 77.2507,
        "metro": "Jor Bagh (Yellow Line) / JLN Stadium (Violet Line)",
        "last_mile": "🛺 Pre-paid Auto or Cab (10 mins ride)",
        "overview": "The first grand garden-tomb on the Indian subcontinent, which inspired the design of Taj Mahal."
    },
    {
        "name": "India Gate & National War Memorial",
        "builder": "Sir Edwin Lutyens",
        "year": "1931",
        "era": "British Raj",
        "lat": 28.6129, "lon": 77.2295,
        "metro": "Central Secretariat (Yellow/Violet Line)",
        "last_mile": "🚶 Easy walking distance through lawns (8-10 mins)",
        "overview": "A soaring 42-meter triumphal arch memorial dedicated to brave soldiers of British India."
    },
    {
        "name": "Lotus Temple (Bahá'í House of Worship)",
        "builder": "Fariborz Sahba",
        "year": "1986",
        "era": "Modern",
        "lat": 28.5535, "lon": 77.2588,
        "metro": "Kalkaji Mandir (Violet/Magenta Line)",
        "last_mile": "🚶 Safe sidewalk pedestrian walk (5 mins)",
        "overview": "An stunning architectural wonder designed as a blooming lotus, inviting members of all religions to pray."
    },
    {
        "name": "Akshardham Temple",
        "builder": "BAPS Swaminarayan Sanstha",
        "year": "2005",
        "era": "Modern",
        "lat": 28.6127, "lon": 77.2773,
        "metro": "Akshardham (Blue Line)",
        "last_mile": "🚶 3-minute walk to the entrance check-post",
        "overview": "An enormous cultural complex showcasing Indian spirituality, fine arts, and ancient architecture."
    },
    {
        "name": "Jama Masjid",
        "builder": "Shah Jahan",
        "year": "1656",
        "era": "Mughal",
        "lat": 28.6507, "lon": 77.2334,
        "metro": "Jama Masjid (Violet Line) / Chawri Bazar (Yellow Line)",
        "last_mile": "🚶 Walk through historic Chandni Chowk alleyways",
        "overview": "One of the grandest mosques in India, looking over the bustling bazaars of Old Delhi."
    },
    {
        "name": "Agrasen ki Baoli",
        "builder": "King Agrasen (Rebuilt later)",
        "year": "14th Century",
        "era": "Tughlaq/Lodi",
        "lat": 28.6261, "lon": 77.2250,
        "metro": "Janpath (Violet Line) / Rajiv Chowk (Blue/Yellow Line)",
        "last_mile": "🚶 Scenic quiet city walk of 8 minutes",
        "overview": "A unique 60-meter long stepped deep well protected right in the heart of modern high-rises."
    },
    {
        "name": "Lodhi Gardens",
        "builder": "Sayyid & Lodi Dynasties",
        "year": "15th Century",
        "era": "Sayyid and Lodi",
        "lat": 28.5900, "lon": 77.2198,
        "metro": "Jor Bagh (Yellow Line) / Lok Kalyan Marg (Yellow Line)",
        "last_mile": "🚶 10-minute stroll or short cycle rickshaw ride",
        "overview": "A serene heritage park dotted with tombs, domed structures, and highly popular walking trails."
    },
    {
        "name": "Jantar Mantar",
        "builder": "Maharaja Sawai Jai Singh II",
        "year": "1724",
        "era": "Rajput",
        "lat": 28.6271, "lon": 77.2166,
        "metro": "Patel Chowk / Rajiv Chowk (Yellow Line)",
        "last_mile": "🚶 Walkable within 5 minutes",
        "overview": "An astronomical observatory with huge stone-built instruments to measure stars and calendar patterns."
    },
    {
        "name": "Safdarjung Tomb",
        "builder": "Nawab Shuja-ud-Daulah",
        "year": "1754",
        "era": "Late Mughal",
        "lat": 28.5893, "lon": 77.2106,
        "metro": "Jor Bagh (Yellow Line)",
        "last_mile": "🚶 2-minute direct crossing",
        "overview": "The last enclosed garden-tomb constructed in the fading grandeur of the Mughal style."
    },
    {
        "name": "Purana Qila (Old Fort)",
        "builder": "Sher Shah Suri / Humayun",
        "year": "1545",
        "era": "Sur Dynasty / Mughal",
        "lat": 28.6096, "lon": 77.2437,
        "metro": "Supreme Court / Pragati Maidan (Blue Line)",
        "last_mile": "🛺 Auto or Cab (5-minute ride)",
        "overview": "Ancient stone ramparts harboring legends associated with the mythical city of Indraprastha."
    },
    {
        "name": "Chandni Chowk & Gurudwara Sis Ganj Sahib",
        "builder": "Shah Jahan / Baghel Singh",
        "year": "1650 / 1783",
        "era": "Mughal / Sikh Era",
        "lat": 28.6565, "lon": 77.2301,
        "metro": "Chandni Chowk (Yellow Line)",
        "last_mile": "🚶 Walk out of exit directly into the busy market street",
        "overview": "The historic central street of Old Delhi hosting dense cultural shops and sacred gurudwaras."
    },
    {
        "name": "Rashtrapati Bhavan & Amrit Udyan",
        "builder": "Sir Edwin Lutyens",
        "year": "1929",
        "era": "British Raj",
        "lat": 28.6143, "lon": 77.1998,
        "metro": "Central Secretariat (Yellow/Violet Line)",
        "last_mile": "🛺 Shuttle tour or walking pathway (10 mins)",
        "overview": "The stately presidential palace of India, famed for its massive Neoclassical design."
    },
    {
        "name": "Hauz Khas Village & Fort Complex",
        "builder": "Alauddin Khalji / Firoz Shah Tughlaq",
        "year": "1300s",
        "era": "Delhi Sultanate",
        "lat": 28.5494, "lon": 77.2036,
        "metro": "IIT (Magenta Line) / Green Park (Yellow Line)",
        "last_mile": "🛺 Auto-Rickshaw directly to the HKV gate",
        "overview": "An antique university ruins, lake, and tombs nestled against an trendy modern lifestyle village."
    },
    {
        "name": "Hazrat Nizamuddin Dargah",
        "builder": "Sufi Disciples",
        "year": "14th Century",
        "era": "Delhi Sultanate",
        "lat": 28.5918, "lon": 77.2415,
        "metro": "JLN Stadium (Violet Line) / Nizamuddin (Pink Line)",
        "last_mile": "🚲 Cycle-Rickshaw or e-rickshaw through lanes",
        "overview": "The soul-stirring shrine of Hazrat Nizamuddin Auliya, world-renowned for its live Sufi Qawwali nights."
    },
    {
        "name": "Laxminarayan Temple (Birla Mandir)",
        "builder": "Baldeo Das Birla",
        "year": "1939",
        "era": "Modern",
        "lat": 28.6328, "lon": 77.2009,
        "metro": "RK Ashram Marg (Blue Line)",
        "last_mile": "🛺 Quick 5-minute Auto ride",
        "overview": "A magnificent multi-tiered red-yellow sandstone Hindu temple inaugurated by Mahatma Gandhi."
    },
    {
        "name": "Chattarpur Temple (Shree Adya Katyayani Shaktipeeth)",
        "builder": "Baba Sant Nagpal Ji",
        "year": "1974",
        "era": "Modern",
        "lat": 28.5021, "lon": 77.1780,
        "metro": "Chhattarpur (Yellow Line)",
        "last_mile": "🚶 Just a 3-minute walk out of metro station",
        "overview": "An incredible, sprawling marble temple complex dedicated to Goddess Katyayani."
    },
    {
        "name": "National Crafts Museum",
        "builder": "Ministry of Textiles",
        "year": "1956",
        "era": "Modern",
        "lat": 28.6136, "lon": 77.2425,
        "metro": "Supreme Court (Blue Line)",
        "last_mile": "🚶 8-minute comfortable walk along Pragati Maidan",
        "overview": "A beautiful village-themed museum displaying rare Indian handicrafts and folk art styles."
    },
    {
        "name": "Gurudwara Bangla Sahib",
        "builder": "Sardar Baghel Singh",
        "year": "1783",
        "era": "Sikh Era",
        "lat": 28.6264, "lon": 77.2091,
        "metro": "Patel Chowk / Rajiv Chowk (Yellow Line)",
        "last_mile": "🚶 Easy 6-minute street walk",
        "overview": "A peaceful sacred pool and gurudwara associated with the eighth Guru, Guru Har Krishan."
    }
]

# Web Page Header
st.write("""
<div style="text-align: center; background-color: #0c1a30; padding: 25px; border-radius: 12px; margin-bottom: 20px;">
    <h1 style="color: #e2f1ff; margin-bottom: 5px;">🕌 Delhi Darshan</h1>
    <p style="color: #a5d0f5; font-size: 1.1rem;">AI Tour Guide & Smart City Companion</p>
</div>
""", unsafe_allow_html=True)

# Sidebar Navigation
st.sidebar.image("https://images.unsplash.com/photo-1585123334904-845d60e97b29?auto=format&fit=crop&q=80&w=400", caption="Discover Historical Delhi", use_container_width=True)
st.sidebar.markdown("### 🗺️ Search & Select Monument")
search_term = st.sidebar.text_input("Quick Search:", "")

# Dynamic Filtering of 20 Monuments
filtered_monuments = [
    m for m in monuments_db if search_term.lower() in m["name"].lower() or search_term.lower() in m["era"].lower()
]
if not filtered_monuments:
    filtered_monuments = monuments_db

selected_mon_name = st.sidebar.selectbox("Choose a Monument:", [m["name" ] for m in filtered_monuments])
selected_mon = next(m for m in monuments_db if m["name"] == selected_mon_name)

# Columns for Dashboard
col1, col2 = st.columns([2, 1])

with col1:
    # Hero Header
    st.markdown(f"## 🏰 {selected_mon['name']}")
    st.markdown(f"*Built around Year **{selected_mon['year']}** by **{selected_mon['builder']}** ({selected_mon['era']} Era)*")
    st.info(f"**Overview:** {selected_mon['overview']}")

    # AI Generated Guided Section
    st.markdown("### 📖 Interactive AI Audio & Narratives")
    with st.spinner("Syncing with local Ollama Guide..."):
        narrative = generate_monument_insights(selected_mon["name"], selected_mon["era"], selected_mon["overview"])

    # Play Audio Guide
    audio_bytes = text_to_mp3_bytes(narrative)
    if audio_bytes:
        st.audio(audio_bytes, format="audio/mp3")
        st.caption("🔊 Click the play button to start the voice-guided tour narration.")

    # Detailed Tab Views
    st.markdown(narrative, unsafe_allow_html=True)

with col2:
    # Weather Card
    st.markdown("### ☀️ Live Smart-City Weather")
    w = get_live_weather(selected_mon["lat"], selected_mon["lon"])
    weather_card = f"""
    <div style="background-color: #1a2d42; padding: 15px; border-radius: 8px; border-left: 5px solid #2196F3;">
        <h4 style="margin: 0; color: #ffffff;">{w['description']}</h4>
        <p style="font-size: 2rem; font-weight: bold; margin: 5px 0; color: #ffeb3b;">{w['temp']} °C</p>
        <p style="margin: 0; color: #bbdefb;">💧 Humidity: {w['humidity']}% | 💨 Wind: {w['wind_speed']} m/s</p>
        <span style="font-size: 0.8rem; color: #90caf9;">{'*Simulated Real-time Data*' if w['simulated'] else '*Live Bureau API Data*'}</span>
    </div>
    """
    st.markdown(weather_card, unsafe_allow_html=True)
    st.markdown(f"*{get_comfort_recommendation(w['temp'], w['description'])}*")

    # Crowd Estimate (integrating Google Places fallback or key estimation)
    st.markdown("### 🚦 Live Crowd Tracker")
    c_level, c_rec = estimate_crowd_level(place_name=selected_mon["name"])
    st.markdown(f"Current Status: **{c_level}**")
    st.info(f"**Visitor Tip:** {c_rec}")

    # Metro & Last-Mile Connectivity Card
    st.markdown("### 🚇 Metro Transit Connectivity")
    st.markdown(f"""
    <div style="background-color: #213214; padding: 15px; border-radius: 8px; border-left: 5px solid #4CAF50;">
        <p style="margin: 0; color: #a5d6a7; font-weight: bold;">🚇 Nearest Station:</p>
        <p style="margin: 5px 0 10px 0; color: #ffffff;">{selected_mon['metro']}</p>
        <p style="margin: 0; color: #a5d6a7; font-weight: bold;">🛵 Last-Mile Connection:</p>
        <p style="margin: 5px 0 0 0; color: #ffffff;">{selected_mon['last_mile']}</p>
    </div>
    """, unsafe_allow_html=True)

# Custom Tour Guide Chat Companion inside Sidebar
st.sidebar.markdown("--- ")
st.sidebar.markdown("### 💬 Ask Delhi AI Guide")
user_chat_input = st.sidebar.text_input("Ask a question (e.g. Best local market nearby?)")
if user_chat_input:
    with st.sidebar.spinner("Consulting AI companion..."):
        try:
            import ollama as ol
            prompt = f"As an expert tour guide in Delhi, answer this query concisely: {user_chat_input}"
            res = ol.chat(model=os.getenv("OLLAMA_MODEL", "llama3.2"), messages=[{"role": "user", "content": prompt}])
            st.sidebar.markdown(res['message']['content'])
        except Exception:
            st.sidebar.warning("Your AI guide is currently in offline mode. Visit Chandni Chowk for delicious Parathas or Sarojini Nagar for shopping bargains!")

Writing app.py


## 🚀 8. Alternative: Exposing App via Ngrok
We will install `pyngrok` and start an ngrok tunnel to expose the Streamlit application safely.

In [8]:
!pip install pyngrok --quiet

Please retrieve your authtoken from [ngrok dashboard](https://dashboard.ngrok.com/) and replace `'YOUR_NGROK_AUTHTOKEN_HERE'` with your actual token.

In [9]:
import subprocess
import time
import getpass
from pyngrok import ngrok

# Securely prompt for the ngrok authtoken
NGROK_TOKEN = getpass.getpass("3KGyfZBh5pVDoQFQxiO2bMr27VH_7xVeqEfXBgaq8q5YunzXq")
if NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN.strip())

# Start Streamlit in background
subprocess.Popen(['pkill', '-f', 'streamlit'])
time.sleep(2)
subprocess.Popen(['streamlit', 'run', 'app.py'])
time.sleep(4)

# Open a HTTP tunnel on port 8501 (default Streamlit port)
try:
    # Close existing tunnels to avoid conflicts
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)

    public_url = ngrok.connect(8501)
    print("\n=========================================")
    print(f"Your ngrok public URL is: {public_url}")
    print("=========================================")
except Exception as e:
    print("Failed to launch ngrok tunnel:", e)

3KGyfZBh5pVDoQFQxiO2bMr27VH_7xVeqEfXBgaq8q5YunzXq··········

Your ngrok public URL is: NgrokTunnel: "https://playable-postnasal-proxy.ngrok-free.dev" -> "http://localhost:8501"
